# Entraînement QLoRA sur Colab (bloc C)

Ce notebook lance l'entraînement du tuteur (Qwen2.5-3B-Instruct, adaptateurs LoRA) sur un GPU Colab. Exécute les cellules dans l'ordre.

## 1. Vérifier le GPU

Choisis un GPU A100 dans **Exécution > Modifier le type d'exécution**, avant de lancer la cellule.
Les unités de calcul Colab sont limitées : vérifie ton solde dans **Ressources** avant un long entraînement.

In [ ]:
!nvidia-smi
import torch
print(torch.cuda.get_device_name(0))

## 2. Monter Google Drive

La session Colab peut être coupée à tout moment, et **tout ce qui n'est pas sur Drive est perdu**.
On sauvegarde donc les adaptateurs directement dans Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/brainmaxing/adapters'

## 3. Récupérer le code

On clone le dépôt sur la branche du bloc C, puis on se place dans son dossier.

In [ ]:
!git clone --branch bloc-c-entrainement https://github.com/EmineOA/NLP-Transformers.git
%cd NLP-Transformers

## 4. Installer les dépendances

Les versions sont figées dans `requirements-qlora.txt`. Torch est déjà installé sur Colab : on ne le réinstalle pas.

In [ ]:
!pip install -r requirements-qlora.txt

## 5. Test rapide

16 exemples train, 8 exemples val, 2 epochs : quelques minutes pour vérifier que tout tourne avant le vrai lancement.

In [ ]:
!python -m src.training.run_training --quick

## 6. Entraînement

Options :
- `--epochs 3` : nombre de passages complets sur les exemples ;
- `--patience 1` : arrêt anticipé, on s'arrête si la loss de validation ne s'améliore pas pendant 1 epoch ;
- `--learning-rate 1e-4` : pas de correction plus petit que le défaut (2e-4), pour limiter le surapprentissage ;
- `--output-dir` : dossier de sortie, sur Drive pour ne rien perdre.

Le meilleur adaptateur est copié dans `v0.3/best`.

In [ ]:
!python -m src.training.run_training --epochs 3 --patience 1 --learning-rate 1e-4 --output-dir /content/drive/MyDrive/brainmaxing/adapters/v0.3

## 7. Évaluer les verdicts

On mesure la précision du verdict (PARFAIT, PARTIEL...) sur le split val, d'abord avec le modèle de base (`--adapter none`), puis avec l'adaptateur entraîné. C'est cette comparaison qui dit si l'entraînement a servi.

In [ ]:
!python -m src.training.evaluate_verdicts --adapter none --split val --output /content/drive/MyDrive/brainmaxing/eval/eval_v0.3_base.json

In [ ]:
!python -m src.training.evaluate_verdicts --adapter /content/drive/MyDrive/brainmaxing/adapters/v0.3/best --split val --output /content/drive/MyDrive/brainmaxing/eval/eval_v0.3_lora.json

## 8. Historique et réglages

Le CSV donne la loss par epoch ; `config.json` garde tous les réglages utilisés (traçabilité).

In [ ]:
!cat /content/drive/MyDrive/brainmaxing/adapters/v0.3/historique_loss.csv
!cat /content/drive/MyDrive/brainmaxing/adapters/v0.3/config.json

## Pistes d'optimisation

Règles : changer **un seul réglage à la fois**, et toujours comparer la **précision du verdict**, pas seulement la loss.

- learning rate 1e-4 contre 2e-4 ;
- dropout 0.1 (`--lora-dropout 0.1`) contre le défaut 0.05 ;
- r à 8 ou 32 (`--lora-rank`, avec `--lora-alpha` = 2 x r pour garder le même rapport) ;
- le split val ne contient que 2 questions : trop étroit pour conclure, à voir avec le bloc A ;
- passer à un modèle 7B est une décision d'équipe : la baseline doit utiliser le même modèle.